In [3]:
!pip install transformers sentence-transformers faiss-cpu datasets torch PyMuPDF -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 22.9 MB/s eta 0:00:00


In [4]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

In [5]:
#loading dataset
dataset = load_dataset("neural-bridge/rag-dataset-12000", split="train[:500]")  # Using 500 samples for fast speed

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

data/train-00000-of-00001-9df3a936e1f631(…):   0%|          | 0.00/23.1M [00:00<?, ?B/s]

data/test-00000-of-00001-af2a9f454ad1b8a(…):   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

In [6]:
documents = [item['context'] for item in dataset]
questions = [item['question'] for item in dataset]
answers   = [item['answer']   for item in dataset]

print(f"Total documents: {len(documents)}")
print(f"\nSample document:\n{documents[0][:400]}...")

Total documents: 500

Sample document:
Caption: Tasmanian berry grower Nic Hansen showing Macau chef Antimo Merone around his property as part of export engagement activities.
THE RISE and rise of the Australian strawberry, raspberry and blackberry industries has seen the sectors redouble their international trade focus, with the release of a dedicated export plan to grow their global presence over the next 10 years.
Driven by signific...


In [7]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(documents, show_progress_bar=True, convert_to_numpy=True)

#faiss indexing
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"Embedding dimension : {dimension}")
print(f"Documents indexed   : {index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding dimension : 384
Documents indexed   : 500


In [8]:
def retrieve(query, top_k=3):
    """
    Given a query, retrieve the top_k most relevant documents.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = []
    for i, idx in enumerate(indices[0]):
        retrieved_docs.append({
            'rank'    : i + 1,
            'document': documents[idx],
            'distance': distances[0][i]
        })
    return retrieved_docs

test_query = "What is machine learning?"
results = retrieve(test_query, top_k=3)

print(f"Query: {test_query}\n")
for r in results:
    print(f"Rank {r['rank']} (Distance: {r['distance']:.4f})")
    print(f"{r['document'][:200]}...")
    print()

Query: What is machine learning?

Rank 1 (Distance: 1.4553)
Telelogic – Effective Requirements Management
Telelogic – Effective Requirements Management
from Effective Requirements Management
Price: USD 0
View Details about Telelogic
Telelogic – Effective Requi...

Rank 2 (Distance: 1.4958)
By in and studied interdisciplinarity and collaboration across a number of fields, through a number of topics, in a variety of ways and in relation to a range of problems. One field that has routinely...

Rank 3 (Distance: 1.5631)
Product category:
Stepper and Servo Drives, Motors, Controls
News Release from: Lenze | Subject: Servomotors
Edited by the Engineeringtalk Editorial Team on 26 August 2002
Driving forward drug discove...



In [9]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

print("Loading QA model...")
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
model_qa  = DistilBertForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
model_qa.eval()

print("QA model loaded successfully!")

Loading QA model...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

QA model loaded successfully!


In [10]:
def get_answer(question, context):
    """
    Use DistilBERT directly (without pipeline) to extract answer from context.
    """
    inputs = tokenizer(question, context, return_tensors="pt",
                       truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model_qa(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx   = torch.argmax(outputs.end_logits) + 1

    input_ids = inputs["input_ids"][0]
    answer    = tokenizer.decode(input_ids[start_idx:end_idx], skip_special_tokens=True)
    return answer


def rag_answer(query, top_k=3):
    """
    Full RAG Pipeline:
    1. Retrieve relevant documents using FAISS
    2. Combine them as context
    3. Generate answer using DistilBERT
    """
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print(f"{'='*60}")

    retrieved = retrieve(query, top_k=top_k)
    combined_context = " ".join([r['document'] for r in retrieved])[:2000]
    answer = get_answer(query, combined_context)

    print(f"\n Retrieved {top_k} relevant documents")
    print(f"\n Top Retrieved Context (snippet):")
    print(f"  {retrieved[0]['document'][:300]}...")
    print(f"\n Generated Answer : {answer}")
    print(f"{'='*60}\n")

    return answer

In [11]:
def evaluate_rag(num_samples=10):
    """
    Compare RAG answers against ground truth answers from the dataset.
    """
    print("Evaluating RAG system on sample questions...\n")
    correct = 0

    for i in range(num_samples):
        query       = questions[i]
        true_answer = answers[i].strip().lower()

        # Retrieve relevant documents
        retrieved   = retrieve(query, top_k=3)
        context     = " ".join([r['document'] for r in retrieved])[:2000]

        # Use get_answer() instead of qa_pipeline()
        pred_answer = get_answer(query, context).strip().lower()

        match = true_answer in pred_answer or pred_answer in true_answer
        if match:
            correct += 1

        print(f"Q{i+1}: {query}")
        print(f"  Ground Truth : {true_answer}")
        print(f"  Predicted    : {pred_answer}")
        print(f"  Match        : {'Y' if match else 'N'}\n")

    accuracy = correct / num_samples * 100
    print(f"Accuracy on {num_samples} samples: {accuracy:.1f}%")

evaluate_rag(num_samples=10)

Evaluating RAG system on sample questions...

Q1: What is the Berry Export Summary 2028 and what is its purpose?
  Ground Truth : the berry export summary 2028 is a dedicated export plan for the australian strawberry, raspberry, and blackberry industries. it maps the sectors’ current position, where they want to be, high-opportunity markets, and next steps. the purpose of this plan is to grow their global presence over the next 10 years.
  Predicted    : maps the sectors ’ current position
  Match        : N

Q2: What are some of the benefits reported from having access to Self-supply water sources?
  Ground Truth : benefits reported from having access to self-supply water sources include convenience, less time spent for fetching water and access to more and better quality water. in some areas, self-supply sources offer important added values such as water for productive use, income generation, family safety and improved food security.
  Predicted    : convenience, less time spent for 

In [18]:
# Try your own question!
while True:
    user_query = input("\nEnter your question (or 'quit' to exit): ")
    if user_query.lower() == 'quit':
        print("Exiting...")
        break
    rag_answer(user_query, top_k=3)


Enter your question (or 'quit' to exit): who is robert downey jr?

Question: who is robert downey jr?

 Retrieved 3 relevant documents

 Top Retrieved Context (snippet):
  As we come to the end of 2014, we thought it would be quite apt to sum up the year with photos which have inspired us. Not through our eyes, but through the eyes of our good friend and award-winning photographer, Edwin Koo. Accolades under his belt include a runner-up for the Unicef Photo of the Yea...

 Generated Answer : Edwin Koo


Enter your question (or 'quit' to exit): what is NSS

Question: what is NSS

 Retrieved 3 relevant documents

 Top Retrieved Context (snippet):
  How unequal is India? The question is simple, the answer is not.
For some 60 years, the only reliable information about India’s inequality was coming from the annual National Sample Survey conducted from 1951. NSS is one of the most venerable surveys in the world of poverty and income distribution s...

 Generated Answer : one of the most ven